## **Projeto de Aprendizagem Computacional 2**

### Grupo 1: Carlos Pereira, João Monteiro Dias, Maria Santos: 202404276 (pedir ups)

**Objetivo**: treinar um modelo capaz de detectar sopros cardíacos a partir de fonocardiogramas.

**Modelos escolhidos**: CNN2D e RNN 


### **CNN2D** : Arquitetura

In [ ]:
from collections import defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(42)

CLASSES = ["Present", "Unknown", "Absent"]  # 0, 1, 2
METRIC_W = np.array([5.0, 3.0, 1.0])


# =====================================================================
# 1. ARQUITETURA DA CNN2D
# =====================================================================
class MurmurCNN:
    @staticmethod
    def build(frames=153, n_mels=40, n_classes=3):
        inp = keras.Input(shape=(frames, n_mels, 1))

        # Bloco 1: 16 filtros
        x = layers.Conv2D(16, 3, padding="same", activation="relu")(inp)
        x = layers.BatchNormalization(momentum=0.9)(x)
        x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(0.25)(x)

        # Bloco 2: 32 filtros
        x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization(momentum=0.9)(x)
        x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(0.25)(x)

        # Bloco 3: 64 filtros
        x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization(momentum=0.9)(x)
        x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(0.25)(x)

        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(64, activation="relu")(x)
        x = layers.Dropout(0.5)(x)
        out = layers.Dense(n_classes, activation="softmax")(x)

        return keras.Model(inp, out, name="Custom_CNN2D")


## RNN Arquitetura

In [ ]:
parte do carlos


## Pré Processamento 
- extração das features .... joão
- split por paciente 
- normalização por janela...

In [ ]:
meter aqui o essencial 

## Estratégia de Treino

Foi desenhada para resolver os dois maiores desafios deste problema (a deteção de sopros cardíacos da competição CirCor DigiScope): o desbalanceamento severo das classes e o desfasamento entre a unidade de entrada do modelo (janela de áudio) e a unidade de diagnóstico clínico (paciente).

- A CNN 2D aprende características locais (tempo-frequência) analisando cada janela de tempo isolada.  

- Agregação por Paciente (to_patient): agrupa todas as janelas que partilham o mesmo pid e calcula a média das probabilidades previstas para esse doente:

    $$\bar{P}(\text{classe} \mid \text{paciente}) = \frac{1}{K} \sum_{k=1}^K P(\text{classe} \mid \text{janela}_k)$$

- Compensação de Classes Desbalanceadas no Treino (class_weight):   Absent: $\approx 73.8\%$   Present: $\approx 19.0\%$   Unknown: $\approx 7.2\%$. 
Para contrariar o desbalancemanto, calculamos os pesos de classe inversamente proporcionais às contagens do conjunto de treino:
    $$\text{weight}_i = \frac{N_{\text{total}}}{3 \times N_i}$$

- **Otimização Orientada à Métrica da Competição (Weighted Accuracy - WA)** : A métrica oficial da competição PhysioNet atribui recompensas desiguais aos acertos com base no vetor de pesos METRIC_W = [5.0, 3.0, 1.0]:  
 -  Present: peso $5.0$ |  Unknown: peso $3.0$ |  Absent: peso $1.0$

Esta otimização é implementada pelas:

- Regra de Decisão do Doente: mean, ou seja as as probabilidades médias eram ponderadas por este vetor (score = pr * METRIC_W) antes de passarem pelo argmax;

- Callback Customizado (PatientWA): no final de cada epoch, a função on_epoch_end corre a validação completa, agrega por paciente e calcula o WA real da validação (val_wa), permitindo monitorizar a weighted accuracy e não apenas a accuracy por janela;

**Gestão Dinâmica da Convergência (Callbacks)**: O treino utiliza dois mecanismos automáticos para evitar overfitting e garantir que a rede converge de forma estável:   
- ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3): Se o erro de validação (val_loss) não diminuir durante 3 épocas consecutivas, a taxa de aprendizagem (learning rate) é reduzida para metade (passando de $0.001$ para $0.0005$, depois $0.00025$, etc.), permitindo que a descida do gradiente refine os pesos.   
- EarlyStopping(monitor="val_wa", mode="max", patience=8, restore_best_weights=True): vigia diretamente o val_wa ao nível do paciente. Se o modelo passar 8 épocas sem conseguir superar o seu recorde de WA, o treino é interrompido e os pesos da melhor época são automaticamente restaurados para o ficheiro final.

**Estas classes estão definidas no ficheiro 'train_strategy.py'**

------------------------------------------------------------------------------------------------------------------------------------------

## CNN2D: Resultados com a estratégia inicial


In [ ]:

CACHE = Path("../dados")
d_tr = np.load(CACHE / "train.npz")
info_tr = pd.read_csv(CACHE / "train_info.csv", dtype={"pid": str})
Xtr, ytr, ptr = d_tr["cnn2d"], d_tr["y"], info_tr["pid"].values

d_va = np.load(CACHE / "val.npz")
info_va = pd.read_csv(CACHE / "val_info.csv", dtype={"pid": str})
Xva, yva, pva = d_va["cnn2d"], d_va["y"], info_va["pid"].values


In [ ]:
from train_strategy import MurmurTrainer, PatientAggregator
model_baseline = MurmurCNN.build(frames=Xtr.shape[1], n_mels=Xtr.shape[2])
trainer_base = MurmurTrainer(model_baseline, weight_mode="standard")
agg_base = PatientAggregator(strategy="mean", decision="weighted")

trainer_base.train(Xtr, ytr, Xva, yva, pva, aggregator=agg_base, epochs=30)

probs_base = model_baseline.predict(Xva, batch_size=128, verbose=0)
yt_b, yp_b, _ = agg_base.aggregate(probs_base, yva, pva)

# Imprime imediatamente o WA_murmur e a matriz de confusão da Baseline
wa_base, _ = trainer_base.evaluate_and_report(
    Xva, yva, pva,
    aggregator=agg_base,
    split_name="VALIDAÇÃO (BASELINE)",
    optimize_thresholds=False
)


- Treino com 1 split: 

        A_murmur: 0.6778

                              |True Present | True Unknown | True Absent
                Pred Present  |          27 |          4   |       62
                Pred Unknown  |          0  |          6   |       13
                Pred Absent   |          0  |          0   |       30 



## Alterações na estratégia de treino e previsão

**Suavização dos pesos das classes no treino (Smoothed Class Weights) com a raiz quadrada**: Substituir os pesos agressivos pela raiz quadrada da frequência inversa das classes (normalizada para média $1.0$), permitindo que a rede calibre probabilidades mais realistas sem ficar demasiado polarizada, mantendo a ponderação 5:3:1 apenas na avaliação final.

**Agregação dos pacientes: Top-K = 5** : em vez de calcularmos a média de todas as janelas temporais de cada paciente, fazemos a média das 5 janelas do doente que têm probabilidade mais elevada de sopro (Present). Evita que um sopro nítido em poucas gravações/janelas seja diluído por várias janelas sem sinal audível.

**Decisão `unweighted` em vez de `weighted`**: o argmax passa a ser aplicado diretamente às probabilidades do doente, sem as multiplicar por [5, 3, 1]. O WA continua a ser calculado com os pesos 5:3:1 na avaliação.


**Estas alterações encontram-se no ficheiro train_strategy.py e podem ser passadas como argumento na chamada do modelo CNN2D.**




In [ ]:
model_improved = MurmurCNN.build(frames=Xtr.shape[1], n_mels=Xtr.shape[2])
trainer_imp = MurmurTrainer(model_improved, weight_mode="smoothed")
agg_imp = PatientAggregator(strategy="topk", k=5, decision="unweighted")

trainer_imp.train(Xtr, ytr, Xva, yva, pva, aggregator=agg_imp, epochs=30)

probs_imp = model_improved.predict(Xva, batch_size=128, verbose=0)
yt_i, yp_i, _ = agg_imp.aggregate(probs_imp, yva, pva)

wa_imp, _ = trainer_imp.evaluate_and_report(
    Xva, yva, pva,
    aggregator=agg_imp,
    split_name="VALIDAÇÃO (IMPROVED)",
    optimize_thresholds=False
)




## Resultados obtidos com as alterações:

- treino com 1 split:

                    WA_murmur: 0.8296

                                        True Present  True Unknown  True Absent
                            Pred Present            27             5           18
                            Pred Unknown             0             1            1
                            Pred Absent              0             4           86

- treino com k-fold cross-validation, k=5:

                        fold	    seed   n_doentes	     WA_fold	 epochs	    melhor_val_wa_interno	
                        0	     42	       159	     0.7741	     22	            0.7826	
                        1	     42	       160	     0.7533	     26	            0.7323	
                        2	     42	       160	     0.7484	     16	            0.6818	
                        3	     42	       160	     0.7336	     17	            0.8384	
                        4	     42	       161	     0.8479	     21	            0.8232	

                        media			0.7715			
                        dp			0.0451			

#### **O k-fold cross-validation está definido no ficheiro kfold_cv.py**


## **Teste:**

### Utilizamos um ensemble dos 5 modelos treinados na validação cruzada

Para a avaliação final usámos o conjunto de teste (`test.npz`), que nunca foi usado no treino, na validação interna (early stopping) nem na escolha de hiperparâmetros. Os modelos avaliados são os 5 obtidos na validação cruzada estratificada por doente (5 folds, `StratifiedGroupKFold`), todos treinados com a mesma configuração (pesos de classe `[smoothed]`, seed 42).

**Procedimento:**

1. Cada um dos 5 modelos (um por fold) prevê as probabilidades de cada janela do conjunto de teste.
2. As probabilidades são **médias entre os 5 modelos**, janela a janela (ensemble por média de probabilidades).
3. As previsões das janelas são agregadas por doente com **Top-K** (K = 5), e a classe final é o argmax das probabilidades agregadas.
4. Calculamos o WA_murmur (pesos 5:3:1 para Present:Unknown:Absent) e a matriz de confusão ao nível do doente.



In [ ]:
MODEL_PATHS = [
    f"cv/fold{f}/model.keras" for f in range(5)
]
TEST_NPZ = "dados/test.npz"

# Carregar dados de teste e modelos
data = np.load(TEST_NPZ, allow_pickle=True)
X_test = data["cnn2d"]
if X_test.ndim == 3:
    X_test = X_test[..., np.newaxis]
y_test = data["y"]

pids_test = pd.read_csv("dados/test_info.csv")["pid"].values

models = [keras.models.load_model(p) for p in MODEL_PATHS]


#Prever janelas (média dos 5 folds) e agregar por paciente
probs_win = np.mean([m.predict(X_test, batch_size=256, verbose=0) for m in models], axis=0)

agg = PatientAggregator(strategy="topk", k=5, decision="unweighted")
yt, yp, _ = agg.aggregate(probs_win, y_test, pids_test)

#Calcular WA e Matriz de Confusão 
w = METRIC_W[yt]
wa = float(np.sum(w * (yt == yp)) / np.sum(w))

cm = np.zeros((3, 3), dtype=int)
for t, p in zip(yt, yp):
    cm[p, t] += 1

print(f"WA_murmur (Teste - Ensemble 5 Folds): {wa:.4f}\n")
print(pd.DataFrame(cm, index=[f"Pred {c}" for c in CLASSES], columns=[f"True {c}" for c in CLASSES]))

**Resultados CNN2D**
- Pesos suavizados;
- Agregação Top K=5;
- Decisão não ponderada (unweighted).


                            WA_murmur (Teste - Ensemble 5 Folds): 0.7852

                                              True Present  True Unknown  True Absent
                            Pred Present            22             3            6
                            Pred Unknown             0             1            0
                            Pred Absent              5             6           99

